<h1>Lab 2: Point Cloud classification with Graph Neural Networks</h1>

In [12]:
# (optional) Mount your Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

Upload the subset of the ModelNet10 dataset for the lab and unzip it. There will be four files corresponding to the point clouds and class labels for training and testing.

In [13]:
# !unzip dataset.zip

**[TODO]** Import all the python modules you need

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [15]:
# read npy files and print
dataset = 'dataset'
X_train = np.load(dataset + '/train_classes.npy')
y_train = np.load(dataset + '/train_points.npy')
X_test = np.load(dataset + '/test_classes.npy')
y_test = np.load(dataset + '/test_points.npy')
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(1281,) (1281, 2048, 3) (200,) (200, 2048, 3)


**[TODO]** Prepare a torch Dataset

In [16]:
class PointCloudDataset(torch.utils.data.Dataset):
    def __init__(self, points_file, classes_file):
        self.points = np.load(points_file)
        self.classes = np.load(classes_file)

    def __len__(self):
        return len(self.points)

    def __getitem__(self, idx):
        return self.points[idx], self.classes[idx]

    def __len__(self):
        return len(self.points)


**[TODO]** Create a torch dataloader instance for training and one for testing

In [17]:
train_dataset = PointCloudDataset(points_file=dataset + '/train_points.npy', classes_file=dataset + '/train_classes.npy')
test_dataset = PointCloudDataset(points_file=dataset + '/test_points.npy', classes_file=dataset + '/test_classes.npy')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

**[TODO]** Create a function that given a batch of point clouds of size (B,N,3) computes the k-nearest neighbor graph using the Euclidean distance between points and returns its adjacency matrix A of size (B,N,N). Make sure to include self-loops so that all the elements on the diagonal are equal to 1

In [18]:
def create_kNN_graph(batch_points, k):
  """
  Computes the k-nearest neighbor graph for a batch of point clouds.

  Args:
    batch_points: A tensor of shape (B, N, 3) representing a batch of point clouds.
    k: The number of neighbors to consider.

  Returns:
    A tensor of shape (B, N, N) representing the adjacency matrix.
  """

  # extract batch size and number of points
  B, N, _ = batch_points.shape
  # initialize adjacency matrix with zeros : A[b, i, j] = 1 if point j is among the k nearest neighbors of point i, else 0
  A = torch.zeros((B, N, N), device=batch_points.device)
  
  # compute pairwise distances
  dist = torch.cdist(batch_points, batch_points)  # (B, N, N)
  # get indices of k-nearest neighbors (excluding self)
  _, knn_idx = torch.topk(dist, k=k+1, largest=False)

  # set the adjacency matrix entries to 1 for k-nearest neighbors
  for b in range(B):
    for i in range(N):
      A[b, i, knn_idx[b, i]] = 1

  return A

**[TODO]** Create a function that given a batch of adjacency matrices of size (B,N,N) returns a batch of diagonal degree matrices of size (B,N,N)

In [19]:
def create_degree_matrix(batch_adj):
  """
  Computes the diagonal degree matrix for a batch of adjacency matrices.

  Args:
    batch_adj: A tensor of shape (B, N, N) representing a batch of adjacency matrices.

  Returns:
    A tensor of shape (B, N, N) representing the diagonal degree matrix.
  """

  # compute degree for each node
  degree = torch.sum(batch_adj, dim=-1)  # (B, N)
  # create diagonal degree matrix
  D = torch.zeros_like(batch_adj)
  for b in range(batch_adj.shape[0]):
    D[b] = torch.diag(degree[b])

  return D

**[TODO]** Create a torch.nn.Module implementing the GCN graph convolution layer. Do not use the pytorch-geometric library. The layer is provided the adjacency matrix A and the degree matrix D. You can call the Linear layer for the multiplication with the weights matrix. **Be careful with dimensions**: input x will have size (B,N,F'), output will be (B,N,F).

In [20]:
class GCNLayer(torch.nn.Module):
  """
  Graph Convolutional Layer.

  Args:
    in_features: Number of input features.
    out_features: Number of output features.
  """
  def __init__(self, in_features, out_features):
    super(GCNLayer, self).__init__()
    self.linear = torch.nn.Linear(in_features, out_features)

  def forward(self, x, A, D):
    """
    Forward pass of the GCN layer.

    Args:
      x: Input features of shape (B, N, F').
      A: Adjacency matrix of shape (B, N, N).
      D: Degree matrix of shape (B, N, N).

    Returns:
      Output features of shape (B, N, F).
    """

    # core equation : H = D^{-1/2} A D^{-1/2} X W, where:
    # - W is the learnable weight matrix of the linear layer
    # - X is the input feature matrix
    # - A is the adjacency matrix
    # - D is the degree matrix

    # 1) Build D^{-1/2} from the diagonal of D
    # extract the diagonal of D to get the degree of each node
    deg = torch.diagonal(D, dim1=-2, dim2=-1) # (B, N)
    # compute D^{-1/2} by taking the reciprocal of the square root of the degree
    deg_inv_sqrt = torch.pow(deg.clamp(min=1e-12), -0.5)  # avoid division by 0
    # create diagonal matrix D^{-1/2}
    D_inv_sqrt = torch.diag_embed(deg_inv_sqrt) # (B, N, N)

    # 2) Normalize adjacency
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt  # (B, N, N)

    # 3) Linear transform
    x = self.linear(x)  
    # Aggregate features from neighbors
    output = A_norm @ x  # (B, N, F_out)

    return output

**[TODO]** Create a torch.nn.Module with your graph neural network for point cloud classification. Compute the A and D matrices from the input point cloud. Use the GCNLayers you previously defined.

In [21]:
class PointCloudGNN(torch.nn.Module):
    def __init__(self, num_classes, k):
        super(PointCloudGNN, self).__init__()
        self.k = k
        self.num_classes = num_classes

        self.gcn1 = GCNLayer(in_features=3, out_features=64)
        self.gcn2 = GCNLayer(in_features=64, out_features=64)

        self.classifier = torch.nn.Linear(64, num_classes)  # Final classifier layer

    def forward(self, x):
        # Forward pass through the GCN layers
        B, N, _ = x.shape

        # 1) Create kNN graph
        A = create_kNN_graph(x, self.k)  # (B, N, N)
        D = create_degree_matrix(A)       # (B, N, N)

        # 2) First GCN layer
        x = self.gcn1(x, A, D)  # (B, N, 64)
        x = torch.nn.functional.relu(x)

        # 3) Second GCN layer
        x = self.gcn2(x, A, D)  # (B, N, 64)
        x = torch.nn.functional.relu(x)

        # 4) Global average pooling
        # for point cloud classification, we can use a global pooling layer followed by a linear layer to get the final class scores
        x = torch.mean(x, dim=1)  # (B, 64)

        # 5) Final classification layer
        output = self.classifier(x)  # (B, num_classes)

        return output

**[TODO]** Create an instance of your neural network

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gcn_model = PointCloudGNN(num_classes=2, k=20)
gcn_model.to(device)

optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=2e-4)
criterion = torch.nn.CrossEntropyLoss()
num_epochs = 20

**[TODO]** Train the network

In [23]:
for epoch in range(num_epochs):
    gcn_model.train()
    total_loss = 0
    for batch_points, batch_labels in train_loader:
        batch_points = batch_points.to(device).float()  # (B, N, 3)
        batch_labels = batch_labels.to(device).long()   # (B,)

        optimizer_gcn.zero_grad()   # reset gradients
        outputs = gcn_model(batch_points)  # (B, num_classes)
        loss = criterion(outputs, batch_labels)
        loss.backward() # compute gradients
        optimizer_gcn.step()    # update parameters

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# save the trained model
torch.save(gcn_model.state_dict(), 'pointcloud_gnn.pth')

Epoch [1/20], Loss: 0.6804
Epoch [2/20], Loss: 0.6241
Epoch [3/20], Loss: 0.6244
Epoch [4/20], Loss: 0.4677


KeyboardInterrupt: 

**[TODO]** Evaluate the network on the test set

In [ ]:
# load the model for evaluation
gcn_model.load_state_dict(torch.load('pointcloud_gnn.pth'))

torch.no_grad()  # disable gradient computation for evaluation
gcn_model.eval()
correct = 0
total = 0
for batch_points, batch_labels in test_loader:
    batch_points = batch_points.to(device).float()  # (B, N, 3)
    batch_labels = batch_labels.to(device).long()   # (B,)

    outputs = gcn_model(batch_points)  # (B, num_classes)
    _, predicted = torch.max(outputs.data, 1)  # get predicted class
    total += batch_labels.size(0)
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')